<a href="https://colab.research.google.com/github/JUST-ZARI/hotel-clv-prediction/blob/main/notebooks/02_harmonize_dataset2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Get the project files.
# Colab: clone the repo (development branch). Locally: run from the repository root.
import os, sys
if "google.colab" in sys.modules:
    !git clone -b development https://github.com/JUST-ZARI/hotel-clv-prediction.git
    %cd /content/hotel-clv-prediction
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: C:\Projects\hotel-clv-prediction-1


### Importing libraries and setting paths

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

RAW_PATH = "data/raw/Hotel Reservations.csv"
OUT_PATH = "data/processed/dataset2_harmonized.csv"

## Load raw data

In [ ]:
df2 = pd.read_csv(RAW_PATH)
print("Shape:", df2.shape)
df2.head(3)

Shape: (36275, 19)


,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled


## Column mapping

In [ ]:
# Rename columns to the common schema {raw_name: harmonized_name}
COLUMN_MAP = {
    "Booking_ID": "booking_id",
    "no_of_adults": "num_adults",
    "no_of_children": "num_children",
    "no_of_weekend_nights": "stays_weekend_nights",
    "no_of_week_nights": "stays_week_nights",
    "type_of_meal_plan": "meal_plan",
    "required_car_parking_space": "required_car_parking_spaces",
    "room_type_reserved": "room_type_reserved",
    "lead_time": "lead_time",
    "arrival_year": "arrival_year",
    "arrival_month": "arrival_month",
    "arrival_date": "arrival_day",
    "market_segment_type": "market_segment",
    "repeated_guest": "is_repeated_guest",
    "no_of_previous_cancellations": "previous_cancellations",
    "no_of_previous_bookings_not_canceled": "previous_bookings_not_canceled",
    "avg_price_per_room": "adr",
    "no_of_special_requests": "total_special_requests",
    "booking_status": "is_canceled",   # re-encoded to 0/1 below
}

df2 = df2.rename(columns=COLUMN_MAP)
df2.columns.tolist()

['booking_id',
 'num_adults',
 'num_children',
 'stays_weekend_nights',
 'stays_week_nights',
 'meal_plan',
 'required_car_parking_spaces',
 'room_type_reserved',
 'lead_time',
 'arrival_year',
 'arrival_month',
 'arrival_day',
 'market_segment',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'adr',
 'total_special_requests',
 'is_canceled']

#Fixing data types

In [ ]:
# Convert binary columns to integers
df2["is_canceled"] = (df2["is_canceled"] == "Canceled").astype(int)
df2["is_repeated_guest"] = df2["is_repeated_guest"].astype(int)
# Fix numeric columns to integers
df2["required_car_parking_spaces"] = df2["required_car_parking_spaces"].astype(int)
df2["arrival_month"] = df2["arrival_month"].astype(int)
df2["arrival_year"] = df2["arrival_year"].astype(int)

# To remember -(Identify where the records came from)
df2["source_dataset"] = "hotel_reservations_2017_2018"

df2.dtypes

booking_id                            str
num_adults                          int64
num_children                        int64
stays_weekend_nights                int64
stays_week_nights                   int64
meal_plan                             str
required_car_parking_spaces         int64
room_type_reserved                    str
lead_time                           int64
arrival_year                        int64
arrival_month                       int64
arrival_day                         int64
market_segment                        str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
adr                               float64
total_special_requests              int64
is_canceled                         int64
source_dataset                        str
dtype: object

## Translate category values into Dataset 1's wording
Renaming columns only aligns the headers. The values inside two columns use different words for the same thing, so they are translated here; otherwise the merged data would treat them as different categories.

| Column | Dataset 2 value | Dataset 1 value |
|---|---|---|
| meal_plan | Meal Plan 1 | BB (bed & breakfast) |
| meal_plan | Meal Plan 2 | HB (half board) |
| meal_plan | Meal Plan 3 | FB (full board) |
| meal_plan | Not Selected | SC (self catering / no meal) |
| market_segment | Online | Online TA |
| market_segment | Offline | Offline TA/TO |

`Corporate`, `Complementary` and `Aviation` are already worded the same.

**Not mapped:** `room_type_reserved` (Dataset 1 uses letters A–L, Dataset 2 uses `Room_Type 1–7`). The codes are anonymised and belong to different hotels, so there is no reliable one-to-one mapping.

In [ ]:
MEAL_PLAN_MAP = {"Meal Plan 1": "BB", "Meal Plan 2": "HB", "Meal Plan 3": "FB", "Not Selected": "SC"}
MARKET_SEGMENT_MAP = {"Online": "Online TA", "Offline": "Offline TA/TO"}

df2["meal_plan"] = df2["meal_plan"].replace(MEAL_PLAN_MAP)
df2["market_segment"] = df2["market_segment"].replace(MARKET_SEGMENT_MAP)

# Every value must now be one Dataset 1 also uses — stop here if not.
DS1_MEAL_PLANS = {"BB", "HB", "FB", "SC", "Undefined"}
DS1_MARKET_SEGMENTS = {"Online TA", "Offline TA/TO", "Groups", "Direct", "Corporate",
                       "Complementary", "Aviation", "Undefined"}
assert set(df2["meal_plan"]) <= DS1_MEAL_PLANS, set(df2["meal_plan"]) - DS1_MEAL_PLANS
assert set(df2["market_segment"]) <= DS1_MARKET_SEGMENTS, set(df2["market_segment"]) - DS1_MARKET_SEGMENTS

print(df2["meal_plan"].value_counts(), "\n")
print(df2["market_segment"].value_counts())

meal_plan
BB    27835
SC     5130
HB     3305
FB        5
Name: count, dtype: int64

 

market_segment
Online TA        23214
Offline TA/TO    10528
Corporate         2017
Complementary      391
Aviation           125
Name: count, dtype: int64


## Columns that Dataset 1 has but Dataset 2 doesn't
These will be empty (NaN) for Dataset 2 rows after the merge.

In [ ]:
missing_from_ds2 = set([
    "hotel_type", "num_babies", "country", "distribution_channel", "assigned_room_type",
    "booking_changes", "deposit_type", "agent_id", "company_id", "days_in_waiting_list",
    "customer_type", "reservation_status", "reservation_status_date",
])
print(f"{len(missing_from_ds2)} columns present in Dataset 1 only:")
missing_from_ds2

13 columns present in Dataset 1 only:


{'agent_id',
 'assigned_room_type',
 'booking_changes',
 'company_id',
 'country',
 'customer_type',
 'days_in_waiting_list',
 'deposit_type',
 'distribution_channel',
 'hotel_type',
 'num_babies',
 'reservation_status',
 'reservation_status_date'}

## Check remaining nulls

In [ ]:
print("Nulls:", df2.isnull().sum().sum())
print("Duplicate rows:", df2.duplicated().sum())

Nulls: 0
Duplicate rows: 0


## Save harmonized output

In [ ]:
OUT_PATH = "data/processed/dataset2_harmonized.csv"
df2.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

Saved to data/processed/dataset2_harmonized.csv
